#**Research/Learn:
# LLM-specific differences**

**A. Causal language models**

Causal language models are a type of machine learning model that generates text by predicting the next word in a sequence based on the words that came before it. Unlike masked language models which predict missing words in a sentence by analyzing both preceding and succeeding words causal models operate in a unidirectional manner—processing text strictly from left to right or right to left.

These models are called "causal" because they rely on a causal relationship: each word depends only on the words that came before it not on any future words. This approach mimics how humans naturally process language as they read or speak.

* **The training process** for causal language models involves two key steps:
  * Step 1: Tokenization
the input text is broken down into smaller units called tokens, which can be words, subwords or even individual characters. For instance the sentence "The cat sleeps" might be tokenized into ["The", "cat", "sleeps"].

  * Step 2: Next-Word Prediction: during training the model learns to predict the next token in a sequence based on the preceding tokens. It does this by analyzing patterns in large datasets of text. Over time, the model becomes adept at understanding grammar, syntax and context allowing it to generate fluent and meaningful sentences. Once trained causal language models can generate text by iteratively predicting one word at a time.

This step-by-step prediction process demonstrates how causal language models generate fluent and meaningful text by focusing on the sequence of words leading up to the current position.

* **Popular Causal Language Models**:GPT,PaLM,LLaMA

* **Applications of Causal Language Models**

  * Content Creation
  * Chatbots and Virtual Assistants
  * Code Generation
  * Creative Writing
  * Language Translation
  * Education
  * Accessibility


**B. Instruction tuning**
* Instruction tuning is a technique where a pre-trained language model is fine-tuned on instruction response pairs to improve its ability to follow user directions. It focuses on making models more adaptable and usable in real-world applications.
  * Helps the model understand and execute natural language instructions
  * Improves generalization across diverse tasks
  * Aligns responses better with user intent

* **Working of Instruction Tuning:**

  * Step 1: Data Collection: a dataset of instruction-output pairs is curated. These pairs should cover a broad spectrum of tasks, including both simple and complex instructions.

    * Examples include:
      * Instruction:"Translate the following sentence into French."
      * Output:"La phrase traduite en français."
  * Step 2: Model Fine-Tuning: the pre-trained LLM is fine-tuned on this dataset using supervised learning techniques. During training, the model learns to map instructions to appropriate outputs.
  * Step 3: Evaluation and Iteration: after fine-tuning, the model is evaluated on a validation set to assess its ability to follow instructions accurately. If necessary, additional data or rounds of fine-tuning are performed to improve performance.

* **Applications**

  * Customer Service

  * Education

  * Content Creation

  * Healthcare

  * E-commerce

  * Marketing & Advertising



**C. Chat Templates**

* Chat Templates serve as the essential bridge between a raw, pre-trained LLM and a functional conversational agent. While a base model only understands a continuous stream of text, "instruct" or "chat" models need a specific structure to distinguish between different speakers and instructions.
* The Role of Chat Templates: a template wraps the conversation history into a specific format using special tokens. These tokens act as markers that tell the model:

  * System Prompt: Where the "personality" or "rules" of the AI are defined.

  * User Message: Where the human's query begins and ends.

  * Assistant Message: Where the AI is expected to generate its response.

* Common Formats: different model families use distinct templates. Mixing them up (e.g., using a Llama template for a Qwen model) often leads to degraded performance or repetitive loops.
  * ChatML (OpenAI/Qwen): Uses <|im_start|> and <|im_end|> tags.
Llama-3: Uses header tags like <|start_header_id|>user<|end_header_id|>.
  * Jinja2 Implementation: Most modern libraries (like Hugging Face transformers) use Jinja2 templating. This allows developers to pass a list of dictionaries—{"role": "user", "content": "..."}—and the tokenizer automatically injects the correct special tokens for that specific model.

**D. Long Context Handling**

* Long Context Handling refers to the techniques used to allow an LLM to process and "remember" extremely long sequences—ranging from 32k to over 1 million tokens—without crashing the system or losing track of information.


#**Here is how the steps differ between ML/DL and LLM:**

# 1. Data Preparation

* ML/DL: Requires highly curated, labeled datasets specifically for the task (e.g., classifying images, predicting numerical values).

* LLMs: Uses massive, often unlabelled, web-scale datasets (trillions of tokens). The "label" is the next word in the text itself (self-supervision), rather than a human-assigned label.

# 2. Model Development & Training

* ML/DL: Models (e.g., Random Forest, CNNs) are trained on specific hardware for hours or days. The process involves manual feature engineering to tell the model what is important.
* LLMs: Involves massive transformer architectures with billions/trillions of parameters. Training is divided into stages:
  * Pre-training: General language mastery (high cost, massive data).
  * Fine-tuning/SFT: Adapting the model for instructions.
  * Reinforcement Learning (RLHF): Aligning the model with human preferences.

# 3. Feature Engineering
* ML/DL: Often requires manual feature engineering, where engineers identify relevant data characteristics.
* LLMs: Features are automatically learned from raw, unstructured text during training, eliminating manual feature extraction.
# 4. Specialization
* ML/DL: Models are specialized "discriminative" models (e.g., "Is this spam: Yes/No?").
* LLMs: Models are "generative" and general-purpose; a single LLM can summarize, translate, and code simultaneously


#**Actions:** Fine-tune a small open LLM using full parameter fine-tuning on an instruction dataset.

In [1]:
!pip install trl torchao>=0.16.0 --upgrade

In [2]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# 2. DATASET

dataset = load_dataset("fawern/Text-to-sql-query-generation")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/277 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [4]:
print(dataset['train'][0])

{'prompt': ' <s> [INST] You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.\n      Question : Name the home team for carlton away team [/INST] SQL Query : SELECT home_team FROM table_name_77 WHERE away_team = "carlton" </s>'}


In [5]:

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM


model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)



config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [6]:
import re

def format_chat(example):

    text = example["prompt"]

    # lấy phần [INST] ... [/INST]
    inst_text = re.search(
        r"\[INST\](.*?)\[/INST\]",
        text,
        re.S
    ).group(1).strip()

    # tách system + question
    parts = inst_text.split("Question :")

    system_message = parts[0].strip()
    question = parts[1].strip()

    # lấy SQL query
    query = re.search(
        r"SQL Query :(.*?)</s>",
        text,
        re.S
    ).group(1).strip()

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": question
        },
        {
            "role": "assistant",
            "content": query
        }
    ]

    return {"messages": messages}


In [7]:
chat= format_chat(dataset['train'][0])
print(chat)

{'messages': [{'role': 'system', 'content': 'You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.'}, {'role': 'user', 'content': 'Name the home team for carlton away team'}, {'role': 'assistant', 'content': 'SELECT home_team FROM table_name_77 WHERE away_team = "carlton"'}]}


In [8]:
formated_dataset = dataset.map(format_chat)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [9]:
def formatting(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}
Qwen_template_dataset=formated_dataset.map(formatting)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [10]:

print(Qwen_template_dataset['train'][0])

{'prompt': ' <s> [INST] You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.\n      Question : Name the home team for carlton away team [/INST] SQL Query : SELECT home_team FROM table_name_77 WHERE away_team = "carlton" </s>', 'messages': [{'role': 'system', 'content': 'You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.'}, {'role': 'user', 'content': 'Name the home team for carlton away team'}, {'role': 'assistant', 'content': 'SELECT home_team FROM table_name_77 WHERE away_team = "carlton"'}], 'text': '<|im_start|>system\nYou are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.<|im_end|>\n<|im_start|>user\nName the home team for carlton away team<|im_end|>\n<|im_start|>assistant\nSELECT home_team FROM table_name_77 WHERE away_team = "carlton"<|im_end|>\n'}


In [11]:
Qwen_template_dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'messages', 'text'],
        num_rows: 10000
    })
})

In [12]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
    )

tokenized_ds = Qwen_template_dataset.map(tokenize)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [13]:
tokenized_ds = tokenized_ds.map(

    remove_columns=[col for col in tokenized_ds["train"].column_names if col not in ["input_ids", "attention_mask"]]
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [14]:
token= tokenized_ds['train'][0]
token

{'input_ids': [151644,
  8948,
  198,
  2610,
  525,
  264,
  7870,
  3239,
  13823,
  320,
  1318,
  4686,
  1331,
  1470,
  568,
  4615,
  3383,
  374,
  311,
  6923,
  264,
  7870,
  3239,
  504,
  279,
  2661,
  3405,
  13,
  151645,
  198,
  151644,
  872,
  198,
  675,
  279,
  2114,
  2083,
  369,
  1803,
  75,
  777,
  3123,
  2083,
  151645,
  198,
  151644,
  77091,
  198,
  4858,
  2114,
  26532,
  4295,
  1965,
  1269,
  62,
  22,
  22,
  5288,
  3123,
  26532,
  284,
  330,
  6918,
  75,
  777,
  1,
  151645,
  198],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [15]:
print(tokenizer.decode(token['input_ids']))

<|im_start|>system
You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.<|im_end|>
<|im_start|>user
Name the home team for carlton away team<|im_end|>
<|im_start|>assistant
SELECT home_team FROM table_name_77 WHERE away_team = "carlton"<|im_end|>



In [16]:
dataset = tokenized_ds["train"].train_test_split(
    test_size=0.1,
    seed=42
)

In [17]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [18]:
dataset['train'][0]

{'input_ids': [151644,
  8948,
  198,
  2610,
  525,
  264,
  7870,
  3239,
  13823,
  320,
  1318,
  4686,
  1331,
  1470,
  568,
  4615,
  3383,
  374,
  311,
  6923,
  264,
  7870,
  3239,
  504,
  279,
  2661,
  3405,
  13,
  151645,
  198,
  151644,
  872,
  198,
  3838,
  374,
  279,
  220,
  17,
  15,
  16,
  16,
  13369,
  5264,
  323,
  264,
  220,
  17,
  15,
  16,
  15,
  1207,
  37,
  30,
  151645,
  198,
  151644,
  77091,
  198,
  4858,
  330,
  17,
  15,
  16,
  16,
  1,
  4295,
  1965,
  62,
  16,
  17,
  22,
  23,
  22,
  5288,
  330,
  51,
  9783,
  1,
  284,
  364,
  2863,
  495,
  10480,
  1787,
  6,
  3567,
  330,
  17,
  15,
  16,
  15,
  1,
  284,
  364,
  80,
  69,
  6,
  151645,
  198],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
 

In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_sql",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=10,
    save_steps=100,

    eval_strategy="steps",
    eval_steps=100,

    report_to="none"
)
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)



In [29]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],

    data_collator=data_collator
)


In [30]:
trainer.train()

Step,Training Loss,Validation Loss
100,1.198593,1.110574
200,1.012559,1.006902
300,0.950094,0.951882
400,0.966086,0.916232
500,1.012566,0.880221
600,0.624526,0.887930
700,0.603839,0.874548
800,0.598538,0.868016
900,0.549414,0.838813
1000,0.567105,0.830346


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1689, training_loss=0.6446801683054372, metrics={'train_runtime': 8633.3158, 'train_samples_per_second': 3.127, 'train_steps_per_second': 0.196, 'total_flos': 7026998731264512.0, 'train_loss': 0.6446801683054372, 'epoch': 3.0})

It's spend 2h30m for training with 3 epoches, and result average_test_loss:0.32 and val_loss:0.89.

Reason Of Overfitting :The dataset is only 10K samples, so the model tends to learn the training set too closely. Training for too many steps/epochs also leads to this result.

In [37]:
messages = [
    {
        "role": "system",
        "content": " You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question."
    },
    {
        "role": "user",
        "content": "Show all customers from Vietnam"
    }
]

# ===== Apply Qwen chat template =====
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# ===== Tokenize =====
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# ===== Generate =====
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=False
    )

# ===== Decode =====
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)


In [38]:

print(response)

system
 You are a SQL query generator (text-to-sql). Your task is to generate a SQL query from the given question.
user
Show all customers from Vietnam
assistant
SELECT * FROM customers WHERE country = 'Thos'


We can see that the response have hallucinations. Because query require vietnamese customers but assistant response country "Thos"

# Milestone: Basic fine-tuned LLM pushed to HF Hub.

In [31]:
import huggingface_hub as hf_hub
hf_hub.login()

In [32]:
tokenizer.push_to_hub(
    "duyb2207513/qwen-text2sql_full_parameter"
)
model.push_to_hub(
    "duyb2207513/qwen-text2sql_full_parameter"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpkm2jlguu/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nxn81pc/model.safetensors:   0%|          | 2.45MB /  988MB            

CommitInfo(commit_url='https://huggingface.co/duyb2207513/qwen-text2sql_full_parameter/commit/729e7707b211227c7b88ec628490fff3ba07fc00', commit_message='Upload Qwen2ForCausalLM', commit_description='', oid='729e7707b211227c7b88ec628490fff3ba07fc00', pr_url=None, repo_url=RepoUrl('https://huggingface.co/duyb2207513/qwen-text2sql_full_parameter', endpoint='https://huggingface.co', repo_type='model', repo_id='duyb2207513/qwen-text2sql_full_parameter'), pr_revision=None, pr_num=None)